# PEC1 - Generative Adversarial Networks
A partir del conjunto de datos CIFAR10, se debe entrenar una *DCGAN* (*Deep Convolutional Generative Adversarial Network*) para generar nuevas imágenes. Una vez implementada la red, se deberá agregar *experience replay* y comparar los resultados con los obtenidos en la red inicial. Por último, se deberá convertir la red en una GAN condicional. 

## Instrucciones
Se debe contestar a cada apartado de la práctica en el espacio correspondiente de esta plantilla. Los apartados que se deben contestar en lenguaje natural (utilizando las celdas de tipo Markdown), se han resaltado en amarillo. Se debe añadir debajo del texto resaltado la respuesta correspondiente. Cuando el texto subrayado indique algún requisito de la respuesta (por ejemplo en cuanto a longitud de la respuesta), será imprescindible ajustarse a dicho requisito. Los apartados que requieren código en Python deberán contestarse en los espacios de código previstos para ello. Al igual que las respuestas en celdas de tipo markdown, las respuestas de código pueden tener asociados requisitos especiales. Estos espacios se delimitan con la siguiente estructura:
``` python
# INICIO_RESPUESTA
# Requisitos: Elabore una función que siempre devuelva True

def mi_funcion() :
    return True

# FIN_RESPUESTA
```

## Esquema de la plantilla
El esquema de la memoria deberá ser el siguiente:

1. Explicar qué es una *DCGAN* y cómo funciona (haciendo énfasis en explicar las dos partes principales de este tipo de modelos) 
1. Describir brevemente el conjunto de datos inicial (*CIFAR10*)
1. Entrenar una *DCGAN (Deep Convolutional Generative Adversarial Network)* para generar imágenes.
1. Añadir *experience replay*
1. Convertir la red a una *GAN condicional*

En primer lugar vamos a cargar las librerías y funciones de apoyo que se van a utilizar en el código del cuadernillo. Si se quiere utilizar alguna librería adicional se puede añadir dicha librería en este punto o en la celda de código en la que se vaya a utilizar por primera vez.


In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals

import tensorflow as tf
assert tf.__version__ >= "2.0"
from tensorflow import keras

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Reshape, Conv2D, BatchNormalization, Conv2DTranspose, LeakyReLU, Flatten, Dropout
from tensorflow.keras.optimizers import RMSprop, Adam

import sklearn
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Check for GPU
!nvidia-smi

print("GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
# print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


## Consideraciones iniciales

Dado el tiempo de cómputo que exigen este tipo de redes se limitará el análisis de 50 epochs. Es muy recomendable, también que, si no se dispone de una tarjeta gráfica y una configuración compatible con tensorflow (puede comprobarse con la última salida del bloque anterior, si el número de GPUs disponible es 0) se utilice para la realización de esta práctica Google Colab. En este plataforma deberemos configurar el entorno de ejecución para que soporte GPUs, esto puede hacerse a través del menú *Entorno de ejecución* y la opción *Cambiar tipo de entorno de ejecución*, selecconando **GPU** como *Acelerador por hardware* .

In [ ]:
# Definimos las constantes que utilizaremos a lo largo del cuadernillo
batch_size = 128
epochs = 50
latent_dim = 100

Es importante saber que para poder depurar algunos métodos como «fit» debemos cambiar el modo de ejecución. Para ello podemos dar el valor «True» al parámetro «run_eagerly» cuando compilemos el modelo. Cuidado, al cambiar este parámetro la ejecución del código será mucho más lenta. Para más información y consejos de depuración se recomienda utilizar los consejos que se indican en la propia página de Keras (https://keras.io/examples/keras_recipes/debugging_tips/)

## Memoria
**Nombre:** Jaime Ciriaco

**Correo UNED:** jciriaco2@alumno.uned.es

## ¿Qué es una DCGAN?

Una DCGAN (Deep Convolutional Generative Adversarial Network) es un tipo de red generativa que se utiliza para crear imágenes nuevas a partir de ruido aleatorio. La idea general es que el modelo está formado por dos redes que compiten entre sí para ir mejorando poco a poco. Por un lado está el generador, que intenta crear imágenes falsas que parezcan reales y por otro lado está el discriminador, que actúa como una especie de detector e intenta distinguir si una imagen viene del conjunto de datos real o si ha sido inventada por el generador.

Lo interesante es que, al entrenarse juntas, ambas redes se empujan mutuamente a mejorar. Si el discriminador se vuelve muy bueno detectando imágenes falsas, el generador tiene que aprender a hacerlas mejor para engañarlo. Y si el generador mejora, el discriminador tiene que afinar más para no dejarse engañar. En una DCGAN, este proceso se hace usando capas convolucionales, que son especialmente útiles cuando se trabaja con imágenes, ya que permiten captar mejor formas, texturas y patrones espaciales.

## CIFAR10

CIFAR10 es un conjunto de datos famoso en visión por computador. Son imágenes en color de tamaño 32x32 píxeles y esta partido en 10 clases: avión, automóvil, pájaro, gato, ciervo, perro, rana, caballo, barco y camión. En total contiene 60.000 imágenes.

Las imágenes al ser pequeñas, complica el fit aunque pueda parecer un dataset sencillo por la poca cantidad de fotos (Hablando de DL), los objetos aparecen en posiciones distintas y dentro de una misma clase puede haber bastante variación. En vista al ejercicio es un buen dataset para ver hasta qué punto el generador consigue crear imágenes que al menos recuerden a las del conjunto original.

In [ ]:
# En primer lugar vamos a descargar el conjunto de datos
(X_train, Y_train), (X_test, Y_test) = keras.datasets.cifar10.load_data()

In [ ]:
# INICIO_RESPUESTA 0/7
# Requisitos: De como valor de la variable «semilla_aleatoria» los últimos dos dígitos de su documento de identificación
semilla_aleatoria = 11
# FIN_RESPUESTA

In [ ]:
np.random.seed(semilla_aleatoria)
tf.random.set_seed(semilla_aleatoria)

# Vamos a visualizar un subconjunto de las imagenes que hemos cargado
clases = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
plt.figure(figsize=(9.6,7.2))
plt.suptitle("CIFAR-10", size=18)
plt.subplots_adjust(wspace=0.2)

pos = 1
for n in np.random.randint(low=0,high=len(X_train),size=12):
    plt.subplot(3, 4, pos)
    plt.imshow(X_train[n])
    plt.axis('off')
    plt.title(clases[Y_train[n][0]], size=16)
    pos += 1
plt.show()

## Pre-procesado de datos

Antes de entrenar la red, las imágenes de CIFAR10 se han preprocesado para que el entrenamiento sea más adecuado. En primer lugar, se convierten a tipo float32, ya que originalmente están almacenadas como enteros entre 0 y 255. Trabajar directamente con enteros no es lo más conveniente en redes neuronales, así que este cambio es el paso más logico.

Después, los valores de los píxeles se reescalan al intervalo [-1,1]. Esto se hace porque la última capa del generador utiliza la función tanh, que precisamente devuelve valores en ese mismo rango. Si las imágenes reales estuvieran en una escala distinta y las falsas en [-1,1], el discriminador lo tendría demasiado fácil al principio, porque distinguiría unas de otras por la escala y no por la calidad de la imagen. Por tanto, poner ambas en el mismo rango ayuda a que la comparación entre imágenes reales y generadas sea más coherente y a que el entrenamiento sea algo más estable.

In [ ]:
# Dado que no tendremos una estructura de entrenamiento y validación, vamos a juntar todos las imágenes
all_imgs = np.concatenate([X_train, X_test])
all_labels = np.concatenate([Y_train, Y_test])

# INICIO_RESPUESTA 1/7
# Requisitos: Realice el pre-procesado de los datos de entrada teniendo en cuenta las recomendaciones que se indican en el capítulo 17 del libro, guarde el resultado en all_imgs.
all_imgs = all_imgs.astype(np.float32) / 127.5 - 1.0
# FIN_RESPUESTA    

In [ ]:
print(all_imgs.min())
print(all_imgs.mean())
print(all_imgs.max())

### DCGAN Inicial
En este apartado se quiere crear una primera red usando la API secuencial de Keras.  Es importante en este punto revisar las recomendaciones que se presentan en el capítulo 17 del libro:

-----------------------------------------------------------------

En el capítulo 17 del libro se comentan varias recomendaciones para construir una DCGAN de forma razonable. Entre ellas, se propone sustituir las capas de pooling por convoluciones con stride en el discriminador y por convoluciones transpuestas en el generador. También se recomienda usar batch normalization en la mayor parte de la red, utilizar ReLU en el generador salvo en la salida, donde se emplea `tanh`, y usar LeakyReLU en el discriminador.

En esta práctica se siguen bastante de cerca estas recomendaciones, porque tienen sentido para trabajar con imágenes. El generador parte de un vector primario y va construyendo la imagen poco a poco con convoluciones transpuestas, mientras que el discriminador hace el proceso contrario y trata de decidir si la imagen es real o falsa. Además, como la salida del generador usa la funcion tanh, se ha ajustado también el preprocesado de los datos para que las imágenes reales estén en el mismo rango. En resumen, no se trata de copiar mecánicamente la teoría, sino de usarla como guía para montar una arquitectura coherente con lo que pide la práctica.

#### Generador

Para el generador se utilizará la siguiente estructura:

1. Capa densa y reshape a 8 x 8 x 128 
2. Capa de convolución transpuesta con 64 filtros (kernel de 4x4, 2 strides y padding 'same')
3. Capa de convolución transpuesta con 64 filtros (kernel de 4x4, 2 strides y padding 'same')
4. Capa convolucional con 3 filtros (kernel de 3x3, 1 stride, padding 'same' y función de activación 'tanh')

Todas las capas convolucionales transpuestas tienen una función de activación Leaky ReLU (únicamente las transpuestas) y, tras cada capa se aplicará normalización por lotes (BatchNormalization).

In [ ]:
generator = keras.models.Sequential([
    keras.layers.Input(shape=(100,)),  

# INICIO_RESPUESTA 2/
# Requisitos: Añada las capas indicadas para el generador      

keras.layers.Dense(8 * 8 * 128),
keras.layers.Reshape((8, 8, 128)),

keras.layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same"),
keras.layers.BatchNormalization(),
keras.layers.LeakyReLU(),

keras.layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same"),
keras.layers.BatchNormalization(),
keras.layers.LeakyReLU(),

keras.layers.Conv2D(3, kernel_size=3, strides=1, padding="same", activation="tanh")
    
# FIN_RESPUESTA 
])

generator.summary()

#### Discriminador

Para el Discriminador se utilizará la siguiente estructura:

1. Capa convolucional con 32 filtros (kernel de 4x4, 2 strides y padding 'same')
2. Capa convolucional con 64 filtros (kernel de 4x4, 2 strides y padding 'same')
3. Capa convolucioanl con 128 filtros (kernel de 4x4, 2 strides y padding 'same')
4. Capa Densa con función de activación sigmoide

Todas las capas convolucionales tienen una función de activación Leaky ReLU y, posteriormente se aplicarán normalización (BatchNormalization) y Dropout. Salvo la última capa convolucional que no aplicará normalización si no que aplanará el resultado de cara a la última capa densa.

In [ ]:
discriminator = keras.models.Sequential([
# INICIO_RESPUESTA 3/7
# Requisitos: Añada las capas indicadas para el discriminador

keras.layers.Input(shape=(32, 32, 3)),

keras.layers.Conv2D(32, kernel_size=4, strides=2, padding="same"),
keras.layers.LeakyReLU(),
keras.layers.BatchNormalization(),
keras.layers.Dropout(0.4),

keras.layers.Conv2D(64, kernel_size=4, strides=2, padding="same"),
keras.layers.LeakyReLU(),
keras.layers.BatchNormalization(),
keras.layers.Dropout(0.4),

keras.layers.Conv2D(128, kernel_size=4, strides=2, padding="same"),
keras.layers.LeakyReLU(),
keras.layers.Dropout(0.4),

keras.layers.Flatten(),
keras.layers.Dense(1, activation="sigmoid")

# FIN_RESPUESTA 
])

discriminator.summary()

#### Modelo

El entrenamiento de una GAN simple no funciona igual que el de una red normal, porque aquí hay dos modelos entrenándose a la vez, cada uno con un objetivo distinto. Por un lado está el discriminador, que recibe imágenes reales y también imágenes falsas creadas por el generador. Su trabajo consiste en aprender a distinguir unas de otras, es decir, decir qué imágenes parecen auténticas y cuáles no.

Por otro lado está el generador, que parte de ruido aleatorio y trata de convertirlo en imágenes cada vez más creíbles para el otro modelo. Aquí está la gracia del proceso: el generador no aprende comparando directamente sus imágenes con una imagen correcta, sino intentando engañar al discriminador. Es decir, va mejorando porque el discriminador le indica indirectamente sus fallos. Por eso puede verse como dos inteligencias artificiales que compiten entre sí para mejorarse en el proceso: una intenta detectar el engaño y la otra intenta colarlo cada vez mejor. A medida que avanza el entrenamiento, el objetivo es que el generador produzca imágenes más realistas y que el discriminador tenga cada vez más difícil diferenciarlas de las reales hasta llegar a un nivel deseado.

In [ ]:
class GAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super(GAN, self).__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super(GAN, self).compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn
        self.d_loss_metric = keras.metrics.Mean(name="d_loss")
        self.g_loss_metric = keras.metrics.Mean(name="g_loss")

    @property
    def metrics(self):
        return [self.d_loss_metric, self.g_loss_metric]

    def train_step(self, real_images):
        
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        
        generated_images = self.generator(random_latent_vectors)
        
        combined_images = tf.concat([generated_images, real_images], axis=0)
        
        labels = tf.concat(
            [tf.ones((batch_size, 1)), tf.zeros((batch_size, 1))], axis=0
        )

        labels += 0.05 * tf.random.uniform(tf.shape(labels))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)

        self.d_optimizer.apply_gradients(
            zip(grads, self.discriminator.trainable_weights)
        )
        
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))

        misleading_labels = tf.zeros((batch_size, 1))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        self.d_loss_metric.update_state(d_loss)
        self.g_loss_metric.update_state(g_loss)
        return {
            "d_loss": self.d_loss_metric.result(),
            "g_loss": self.g_loss_metric.result(),
        }
    
class GANMonitor(keras.callbacks.Callback):
    def __init__(self, num_img=3, latent_dim=100, show_range=10, enable_show=True, enable_save=False):
        self.num_img = num_img
        self.latent_dim = latent_dim
        self.show_range = show_range
        self.enable_show = enable_show
        self.enable_save = enable_save

    def on_epoch_end(self, epoch, logs=None):
        if ((epoch+1) % self.show_range) == 0 or epoch == 0:
          random_latent_vectors = tf.random.normal(shape=(self.num_img, self.latent_dim))
          generated_images = self.model.generator(random_latent_vectors)
          generated_images += 1
          generated_images /= 2

          generated_images.numpy()
          plt.figure(figsize=(self.num_img, 1))
          for i in range(self.num_img):
              img = keras.preprocessing.image.array_to_img(generated_images[i])
              if self.enable_show:
                plt.subplot(1, self.num_img, i + 1)
                plt.imshow(img, cmap="binary")
                plt.axis("off")
              if self.enable_save:
                img.save("generated_img_%03d_%d.png" % (epoch, i))
          if self.enable_show:
            plt.show()

In [ ]:
def plt_loss(history): 
    '''
    Función que nos ayudará a visualizar la evolución de las pérdidas del generador y del discriminador
    '''
    f_num = "{:.3f}"
    plt.figure(figsize=(12,5))
    plt.suptitle("Función de pérdida", size=18)
    plt.subplots_adjust(wspace=0.5, hspace = 0.5)
    plt.subplot(1, 2, 1)
    plt.plot(history.history['d_loss'])
    plt.title("Discriminador")
    plt.ylabel('Loss')
    plt.xlabel('Epoch')

    plt.subplot(1, 2, 2)
    plt.plot(history.history['g_loss'])
    plt.title("Generador")
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.tight_layout(pad=4)
    plt.show()
    print(f"Pérdida discriminador: Min ({f_num.format(np.min(history.history['d_loss']))}) Max ({f_num.format(np.max(history.history['d_loss']))}) Media ({f_num.format(np.mean(history.history['d_loss']))})")
    print(f"Pérdida generador: Min ({f_num.format(np.min(history.history['g_loss']))}) Max ({f_num.format(np.max(history.history['g_loss']))}) Media ({f_num.format(np.mean(history.history['g_loss']))})")

#### Optimizadores y función de pérdida

Para entrenar tanto el generador como el discriminador se utiliza el optimizador Adam. En este tipo de redes suele funcionar bien usar una tasa de aprendizaje baja, ya que el entrenamiento de una GAN es bastante delicado y pequeños cambios pueden desestabilizar el proceso. Por eso se ha usado un valor en torno a 0.0002, que además coincide con lo que suele recomendarse en implementaciones básicas de DCGAN.

En cuanto a la función de pérdida, se utiliza la binary crossentropy. Tiene sentido porque el discriminador está resolviendo un problema binario: decidir si una imagen es real o falsa. Esa misma idea se aprovecha también al entrenar el generador, ya que su objetivo es crear imágenes que hagan que el discriminador se equivoque y las marque como reales. No es una métrica perfecta para entender por sí sola la calidad visual del modelo, pero sí es la más habitual para entrenar este tipo de GANs sencillas.

In [ ]:
# INICIO_RESPUESTA 4/7
# Requisitos: Inicialice los optimizadores que se utilizarán para el discriminador y para el generador, se aconseja utilizar valores LR de aproximadamente 0.0002
d_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
g_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
# FIN_RESPUESTA 

loss_fn = keras.losses.BinaryCrossentropy()

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices(all_imgs).shuffle(1000)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)

gan = GAN(discriminator=discriminator, generator=generator, latent_dim=latent_dim)
gan.compile(
    d_optimizer=d_optimizer,
    g_optimizer=g_optimizer,
    loss_fn=loss_fn,
)

history = gan.fit(dataset, epochs=epochs, callbacks=[GANMonitor(num_img=10, latent_dim=latent_dim)])

In [ ]:
plt_loss(history)

#### Resultados

A partir de las curvas de pérdida se puede ver que el entrenamiento de la DCGAN inicial es relativamente estable, aunque con las oscilaciones normales. En el caso del discriminador, la pérdida se mueve alrededor de valores cercanos a 0.67, sin cambios demasiado bruscos durante la mayor parte del entrenamiento. Esto suele indicar que no está ganando claramente la partida, sino que se mantiene en una situación de equilibrio razonable con el generador.

En el generador ocurre algo parecido. La pérdida empieza siendo más alta, baja con bastante rapidez en las primeras épocas y luego se estabiliza alrededor de valores próximos a 0.80. Esto da a entender que el modelo aprende pronto algunos patrones básicos, aunque después la mejora se vuelve mucho más lenta. En conjunto, los resultados parecen razonables para una primera GAN sobre CIFAR10, que no es precisamente un conjunto fácil. De todos modos, en una GAN no basta con mirar solo las pérdidas: también hay que fijarse en si las imágenes generadas empiezan a parecerse mínimamente a las reales.

## Experience replay

La técnica de experience replay consiste en no entrenar al discriminador únicamente con las imágenes falsas recién generadas en la iteración actual, sino también con algunas imágenes falsas producidas anteriormente y guardadas en un pequeño buffer. La idea es simple, si el discriminador solo ve lo último que hace el generador, puede adaptarse demasiado rápido a ese estado concreto y volverse inestable el entrenamiento. Sinembargo, si también repasa ejemplos anteriores, tiene una visión algo más variada de los errores del generador.

En esta práctica se ha implementado una versión simple de esta idea. Se guardan imágenes falsas generadas en distintas iteraciones y, cuando el buffer ya tiene suficientes ejemplos, se reutiliza una parte de ellos para volver a entrenar al discriminador. No es de las mejores implementaciones, pero sirve para ilustrar bien el concepto. Lo esperable es que esta técnica ayude a que el discriminador no cambie de criterio tan bruscamente y que el entrenamiento sea algo más estable, aunque no siempre tiene por qué traducirse en una mejora muy grande en las pérdidas o en la calidad visual final.

#### Modelo

In [ ]:
class GAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super(GAN, self).__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.experience_replay = []

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super(GAN, self).compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn
        self.d_loss_metric = keras.metrics.Mean(name="d_loss")
        self.g_loss_metric = keras.metrics.Mean(name="g_loss")

    @property
    def metrics(self):
        return [self.d_loss_metric, self.g_loss_metric]

    def train_step(self, real_images):
        
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        
        generated_images = self.generator(random_latent_vectors)
        
        combined_images = tf.concat([generated_images, real_images], axis=0)
        
        labels = tf.concat(
            [tf.ones((batch_size, 1)), tf.zeros((batch_size, 1))], axis=0
        )

        labels += 0.05 * tf.random.uniform(tf.shape(labels))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(
            zip(grads, self.discriminator.trainable_weights)
        )

        # INICIO_RESPUESTA 5/7
        # Requisitos: Implementar una estrategia de Experience replay a partir de un buffer de imágenes alamacenadas en cada iteración

        # Guardamos en el buffer las imágenes falsas generadas en la iteración actual
        for img in tf.unstack(tf.stop_gradient(generated_images)):
            self.experience_replay.append(img)
        
        # Limitamos el tamaño del buffer para que no crezca indefinidamente
        max_buffer_size = 1000
        if len(self.experience_replay) > max_buffer_size:
            self.experience_replay = self.experience_replay[-max_buffer_size:]
        
        # Tomamos una muestra del buffer y volvemos a entrenar el discriminador con esas imágenes falsas
        current_batch_size = real_images.shape[0]
        if current_batch_size is None:
            current_batch_size = int(tf.shape(real_images)[0].numpy())
        
        if len(self.experience_replay) >= current_batch_size:
            idx = np.random.choice(len(self.experience_replay), size=current_batch_size, replace=False)
            replay_images = tf.stack([self.experience_replay[i] for i in idx], axis=0)
        
            replay_labels = tf.ones((current_batch_size, 1))
            replay_labels += 0.05 * tf.random.uniform(tf.shape(replay_labels))
        
            with tf.GradientTape() as tape:
                replay_predictions = self.discriminator(replay_images)
                replay_d_loss = self.loss_fn(replay_labels, replay_predictions)
        
            replay_grads = tape.gradient(replay_d_loss, self.discriminator.trainable_weights)
            self.d_optimizer.apply_gradients(
                zip(replay_grads, self.discriminator.trainable_weights)
            )

        d_loss = 0.5 * (d_loss + replay_d_loss)

        # FIN_RESPUESTA 
        
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))

        misleading_labels = tf.zeros((batch_size, 1))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        # Update metrics
        self.d_loss_metric.update_state(d_loss)
        self.g_loss_metric.update_state(g_loss)
        return {
            "d_loss": self.d_loss_metric.result(),
            "g_loss": self.g_loss_metric.result(),
        }

In [ ]:
gan_er = GAN(discriminator=discriminator, generator=generator, latent_dim=latent_dim)
gan_er.compile(
    d_optimizer=d_optimizer,
    g_optimizer=g_optimizer,
    loss_fn=loss_fn,
)

history = gan_er.fit(dataset, epochs=epochs, callbacks=[GANMonitor(num_img=10, latent_dim=latent_dim)])

In [ ]:
plt_loss(history)

#### Resultados

Al comparar esta versión con experience replay con la GAN inicial, no se aprecia un cambio radical en las pérdidas, pero sí puede notarse un comportamiento algo más suave o menos brusco en algunos tramos del entrenamiento. La idea de esta técnica es que el discriminador no aprenda solo de las imágenes falsas más recientes, sino también de algunas generadas en iteraciones anteriores. Dicho de forma sencilla, en vez de “olvidar” enseguida lo que veía hace un momento, mantiene una pequeña memoria de ejemplos falsos pasados.

En principio, esto debería ayudar a que el discriminador no se adapte demasiado rápido a los cambios del generador y que el entrenamiento sea algo más estable. Aun así, en una GAN este tipo de mejora no siempre se traduce en una bajada clara de la pérdida, porque ambas redes están compitiendo constantemente y las curvas suelen oscilar. Por eso, más que esperar una diferencia enorme en los números, lo razonable es comprobar si las imágenes generadas parecen algo más variadas, si hay menos cambios bruscos en el entrenamiento y si el generador consigue mantener mejor lo aprendido respecto a la GAN sin experience replay.

### GAN condicional

Una GAN condicional es una extensión de la GAN original en la que, además del ruido aleatorio, se introduce también información sobre la clase que se quiere generar. En lugar de dejar que el modelo invente una imagen cualquiera, se le da una especie de pista inicial. Por ejemplo, no se le pide solo que genere una imagen realista, sino que genere una imagen realista de una clase concreta, como un avión, un perro o un barco.

En relación con el modelo original y con la versión con experience replay, lo que se espera aquí no es necesariamente una gran mejora en las pérdidas, sino una mejora en el control de la generación. Es decir, el valor añadido de esta versión está en que permite orientar el resultado hacia una categoría determinada. A cambio, el problema también se vuelve un poco más exigente, porque el generador no solo tiene que “engañar” al discriminador, sino hacerlo generando algo coherente con la clase indicada. Por eso, aunque las pérdidas puedan ser parecidas o incluso algo peores que en la GAN base, esta variante resulta más útil desde el punto de vista práctico, ya que no genera imágenes al azar, sino imágenes condicionadas por la etiqueta deseada.

#### Discriminador

Para convertir la GAN en condicional, el discriminador ya no puede limitarse a decidir si una imagen es real o falsa sin más. Ahora también tiene que tener en cuenta la clase asociada a esa imagen. Es decir, no solo debe valorar si la imagen parece realista, sino también si encaja con la etiqueta que se le está dando. Por ejemplo, no sería lo mismo recibir una imagen parecida a un barco con la etiqueta de “ship” que con la etiqueta de “cat”.

Para conseguir esto, se añade como entrada extra la información de la clase codificada en formato *one-hot*. Esa información se adapta y se combina con la imagen antes de pasar por las capas convolucionales. De esa forma, el discriminador puede aprender a relacionar el contenido visual con la clase esperada. Dicho de forma sencilla, deja de preguntarse solo “¿esto parece real?” y pasa a preguntarse también “¿esto parece real y además corresponde a la clase que me dicen?”.

In [ ]:
# INICIO_RESPUESTA 6/7
# Requisitos: Implemente el discriminador para la GAN condicional, utilice la estructura del discriminador original y adáptela para que funcione con las etiquetas de clase
#             (utilice la variable «discriminator» para almacenar el discriminador tal y como se hizo en el caso base)
# Nota: se recomienda utilizar la API funcional para desarrollar este apartado.

image_input = keras.layers.Input(shape=(32, 32, 3))
label_input = keras.layers.Input(shape=(10,))

label_projection = keras.layers.Dense(32 * 32)(label_input)
label_projection = keras.layers.Reshape((32, 32, 1))(label_projection)

x = keras.layers.Concatenate()([image_input, label_projection])

x = keras.layers.Conv2D(32, kernel_size=4, strides=2, padding="same")(x)
x = keras.layers.LeakyReLU()(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.4)(x)

x = keras.layers.Conv2D(64, kernel_size=4, strides=2, padding="same")(x)
x = keras.layers.LeakyReLU()(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.4)(x)

x = keras.layers.Conv2D(128, kernel_size=4, strides=2, padding="same")(x)
x = keras.layers.LeakyReLU()(x)
x = keras.layers.Dropout(0.4)(x)

x = keras.layers.Flatten()(x)
output = keras.layers.Dense(1, activation="sigmoid")(x)

discriminator = keras.Model([image_input, label_input], output)

# FIN_RESPUESTA

#### Generador

En el caso del generador, la adaptación consiste en que ya no parte solo de ruido aleatorio, sino también de la clase que queremos generar. Así, en lugar de crear imágenes de manera totalmente libre, recibe una pista adicional que le indica hacia qué tipo de imagen debe orientarse.

La clase se introduce codificada en one-hot y se combina con el vector latente antes de comenzar la generación. A partir de ahí, la red sigue una estructura parecida a la del caso base, pero ahora con la ventaja de que la salida queda condicionada por esa etiqueta. En la práctica, esto significa que el generador no solo intenta producir una imagen que parezca real, sino que además trata de que esa imagen se parezca a la clase solicitada. Es una forma bastante intuitiva de añadir control al proceso de generación.

In [ ]:
# INICIO_RESPUESTA 7/7
# Requisitos: Implemente el generador para la GAN condicional, utilice la estructura del generador original y adáptela para que funcione con las etiquetas de clase 
#             (utilice la variable «generator» para almacenar el generador tal y como se hizo en el caso base)
# Nota: se recomienda utilizar la API funcional para desarrollar este apartado.

label_input = keras.layers.Input(shape=(10,))
noise_input = keras.layers.Input(shape=(latent_dim,))

x = keras.layers.Concatenate()([label_input, noise_input])

x = keras.layers.Dense(8 * 8 * 128)(x)
x = keras.layers.Reshape((8, 8, 128))(x)

x = keras.layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.LeakyReLU()(x)

x = keras.layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.LeakyReLU()(x)

output = keras.layers.Conv2D(3, kernel_size=3, strides=1, padding="same", activation="tanh")(x)

generator = keras.Model([label_input, noise_input], output)

# FIN_RESPUESTA

#### Modelo

In [ ]:
class GAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super(GAN, self).__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.experience_replay = []

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super(GAN, self).compile(run_eagerly=True)
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn
        self.d_loss_metric = keras.metrics.Mean(name="d_loss")
        self.g_loss_metric = keras.metrics.Mean(name="g_loss")

    @property
    def metrics(self):
        return [self.d_loss_metric, self.g_loss_metric]

    def train_step(self, real_images):
        batch_size = tf.shape(real_images[0])[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))

        generated_images = self.generator([real_images[1],random_latent_vectors])

        combined_images = tf.concat([generated_images, real_images[0]], axis=0)

        labels = tf.concat(
            [tf.ones((batch_size, 1)), tf.zeros((batch_size, 1))], axis=0
        )
        combined_labels = tf.concat([real_images[1], real_images[1]], axis=0)

        labels += 0.05 * tf.random.uniform(tf.shape(labels))

        with tf.GradientTape() as tape:
            predictions = self.discriminator([combined_images, combined_labels])
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(
            zip(grads, self.discriminator.trainable_weights)
        )
        
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))

        misleading_labels = tf.zeros((batch_size, 1))
        
        with tf.GradientTape() as tape:
            predictions = self.discriminator([self.generator([real_images[1],random_latent_vectors]), real_images[1]])
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))


        self.d_loss_metric.update_state(d_loss)
        self.g_loss_metric.update_state(g_loss)
        return {
            "d_loss": self.d_loss_metric.result(),
            "g_loss": self.g_loss_metric.result(),
        }
    
class GANMonitor(keras.callbacks.Callback):
    def __init__(self, y_train_enc = None, num_img=3, latent_dim=100, show_range=10,
                 enable_show=True, enable_save=False,
                 batch_size = 128):
        super().__init__()
        self.num_img = num_img
        self.latent_dim = latent_dim
        self.show_range = show_range
        self.enable_show = enable_show
        self.enable_save = enable_save
        self.batch_size = batch_size
        self.y_train_enc = y_train_enc

    def on_epoch_end(self, epoch, logs=None):
        if ((epoch+1) % self.show_range) == 0 or epoch == 0:
          random_latent_vectors = tf.random.normal(shape=(self.num_img, self.latent_dim))
          labels = self.y_train_enc[epoch:epoch+self.num_img]
          generated_images = self.model.generator([labels,random_latent_vectors])
          generated_images = (generated_images + 1) / 2.0
          
          generated_images.numpy()
          plt.figure(figsize=(self.num_img, 1))
          for i in range(self.num_img):
              img = keras.preprocessing.image.array_to_img(generated_images[i])
              if self.enable_show:
                plt.subplot(1, self.num_img, i + 1)
                plt.imshow(img, cmap="binary")
                plt.axis("off")
              if self.enable_save:
                img.save("generated_img_%03d_%d.png" % (epoch, i))
          if self.enable_show:
            plt.show()

In [ ]:
# Dado que ahora el generador y el discriminador utilizarán la clase de cada elemento, es necesario utilizar las etiquetas, transformándolas previamente a un array
all_labels_enc = tf.convert_to_tensor(keras.utils.to_categorical(all_labels, 10))
dataset = tf.data.Dataset.from_tensor_slices((all_imgs, all_labels_enc)).shuffle(1000)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)

In [ ]:
d_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
g_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)

conditional_gan = GAN(discriminator=discriminator, generator=generator, latent_dim=latent_dim)
conditional_gan.compile(
    d_optimizer=d_optimizer,
    g_optimizer=g_optimizer,
    loss_fn=loss_fn
)

history = conditional_gan.fit(dataset, epochs=epochs, callbacks=[GANMonitor(all_labels_enc,
                                                                num_img=10,
                                                                latent_dim=latent_dim)])

In [ ]:
plt_loss(history)

In [ ]:
classes_labels = tf.convert_to_tensor(keras.utils.to_categorical([x for x in range(10)] * 4, 10))

random_latent_vectors = tf.random.normal(shape=(len(classes_labels), latent_dim))
generated_images = conditional_gan.generator([classes_labels,random_latent_vectors])
generated_images = (generated_images + 1) / 2.0
# generated_images.numpy()
plt.figure(figsize=(20, 8))
plt.subplots_adjust(wspace=0.2, hspace=0.2)
pos = 1
classes_to_int = np.argmax(classes_labels, axis=-1)
for n in range(generated_images.shape[0]):
    plt.subplot(4, 10, pos)
    img = keras.preprocessing.image.array_to_img(generated_images[n])
    plt.imshow(img)
    plt.axis('off')
    plt.title(clases[classes_to_int[n]], size=12)
    pos += 1
plt.show()

#### Resultados

En la GAN condicional se observa un comportamiento parecido al de la red base, aunque con una tarea algo más exigente. La pérdida del discriminador se mantiene bastante estable, en torno a valores cercanos a 0.66-0.68, mientras que la del generador desciende al principio y luego se mueve alrededor de 0.80-0.85. No se aprecia una convergencia perfecta, pero eso es bastante normal en GANs, donde las dos redes están compitiendo constantemente y las curvas rara vez son suaves.

Comparada con la GAN inicial, esta versión condicional no tiene por qué dar pérdidas claramente mejores, porque ahora el problema es más difícil: el generador no solo debe crear imágenes realistas, sino además hacer que correspondan a una clase concreta. Frente a la versión con experience replay, la principal ventaja aquí no es tanto una mejora evidente en las pérdidas, sino el hecho de poder controlar qué tipo de imagen se quiere generar. Por eso, en este caso tiene todavía más importancia mirar también las imágenes finales y no quedarse solo con los números.

### Conclusiones finales

En esta práctica se ha trabajado con CIFAR10 para implementar y comparar tres variantes de redes generativas adversarias: una DCGAN inicial, una segunda versión con experience replay y una tercera versión condicional. A lo largo del trabajo se ha visto que entrenar una GAN no es tan directo como entrenar una red supervisada normal, ya que aquí hay dos modelos compitiendo entre sí y el equilibrio entre ambos es bastante delicado.

Como idea general, la DCGAN base permite comprobar que el generador empieza a aprender ciertos patrones visuales del conjunto de datos, aunque con la inestabilidad típica de este tipo de redes. La variante con experience replay introduce una pequeña memoria de imágenes falsas anteriores para intentar dar algo más de estabilidad al discriminador. Por último, la GAN condicional añade una mejora muy interesante, ya que permite orientar la generación hacia una clase concreta. En conjunto, la práctica deja claro que las GAN son modelos muy potentes, pero también bastante sensibles a detalles de implementación, al número de épocas y a la configuración elegida.

### (Opcional) Cycle-GANs

Las Cycle-GANs son una variante de las GAN pensada para transformar imágenes de un dominio a otro sin necesidad de tener pares exactos entre ellas. Por ejemplo, se pueden usar para pasar de imágenes de caballos a cebras, o de paisajes de verano a paisajes de invierno, sin que haga falta disponer de una versión emparejada de cada imagen. Esto las hace especialmente útiles en problemas donde sí hay ejemplos de ambos tipos de imagen, pero no una correspondencia uno a uno o para ampliarlo a más proyectos reales fuera de un ejercicio como este.

La novedad principal que introducen es la idea de consistencia cíclica. Dicho de forma simple, si una imagen se transforma de A a B y luego se vuelve a transformar de B a A, debería parecerse a la original. Esa restricción ayuda a que la red no haga cambios arbitrarios como si tradujeras un texto 2 veces volviendo al mismo idioma. Frente a lo visto en esta práctica, una Cycle-GAN no se centra tanto en generar imágenes desde ruido ni en controlar una clase concreta, sino en aprender a traducir imágenes entre dos estilos o dominios distintos.